In [66]:
import napari
from qtpy.QtWidgets import QWidget, QVBoxLayout, QHBoxLayout, QPushButton, QLabel, QStackedWidget
from magicgui.widgets import ComboBox, FileEdit

In [67]:
class MyTool(QWidget):

    def __init__(self, viewer):
        super().__init__()

        self.viewer = viewer
        self.state = {}

        # Holds all pages
        self.pages = QStackedWidget()

        self.upload_page = self.create_upload_page()
        self.threshold_page = self.create_threshold_page()
        self.cellpose_page = self.create_cellpose_page()

        self.pages.addWidget(self.upload_page)
        self.pages.addWidget(self.threshold_page)
        self.pages.addWidget(self.cellpose_page)

        layout = QVBoxLayout()
        layout.addWidget(self.pages)
        self.setLayout(layout)

    # --------------------------
    # PAGE 1
    # --------------------------
    def change_upload_type(self):

        is_image = self.upload_type.value == "Image"

        for widgets in self.inputs.values():

            widgets["image"].visible = is_image
            widgets["folder"].visible = not is_image

            if is_image:
                widgets["label"].setText(widgets["image_label"])
            else:
                widgets["label"].setText(widgets["folder_label"])


    def create_upload_page(self):

        page = QWidget()
        layout = QVBoxLayout(page)
        layout.addWidget(QLabel("Upload Your Data"))

        # Image / Folder selector
        self.upload_type = ComboBox(label="Input type:", choices=["Image", "Folder"], value="Image")
        layout.addWidget(self.upload_type.native)

        # Store our input widgets
        self.inputs = {}

        # Add the three rows
        self.add_row(layout, "original", "Original Image (Stack)", "(Folder of) Original Images")
        self.add_row(layout, "labels", "(Stack of) Labels", "(Folder of) Labels")
        self.add_row(layout, "tracks", "(Stack of) Linked Tracks", "(Folder of) Linked Tracks")

        # Update everything when Image/Folder changes
        self.upload_type.changed.connect(self.change_upload_type)

        # Next button
        next_button = QPushButton("Next →")
        # next_button.clicked.connect(self.next_page)
        layout.addWidget(next_button)

        return page


    def add_row(self, layout, name, image_label_text, folder_label_text):
        row = QHBoxLayout()
        label = QLabel(image_label_text)
        image_input = FileEdit(label="", mode="r", filter="TIFF (*.tif *.tiff)")
        folder_input = FileEdit(label="", mode="d")

        # Default to image input, so hide the folder input intiailly 
        folder_input.visible = False

        row.addWidget(label)
        row.addWidget(image_input.native)
        row.addWidget(folder_input.native)

        layout.addLayout(row)

        self.inputs[name] = {
            "label": label,
            "image": image_input,
            "folder": folder_input,
            "image_label": image_label_text,
            "folder_label": folder_label_text
        }


    # --------------------------
    # PAGE 2
    # --------------------------

    def create_threshold_page(self):

        page = QWidget()
        layout = QVBoxLayout(page)

        layout.addWidget(
            QLabel("Configure analysis")
        )

        back = QPushButton("← Back")
        next_button = QPushButton("Next →")

        back.clicked.connect(
            lambda: self.pages.setCurrentIndex(0)
        )

        next_button.clicked.connect(
            lambda: self.pages.setCurrentIndex(2)
        )

        layout.addWidget(back)
        layout.addWidget(next_button)

        return page

    # --------------------------
    # PAGE 3
    # --------------------------

    def create_cellpose_page(self):

        page = QWidget()
        layout = QVBoxLayout(page)

        layout.addWidget(
            QLabel("Ready to run!")
        )

        back = QPushButton("← Back")
        run = QPushButton("Run analysis")

        back.clicked.connect(
            lambda: self.pages.setCurrentIndex(1)
        )

        run.clicked.connect(self.run_analysis)

        layout.addWidget(back)
        layout.addWidget(run)

        return page

    # --------------------------
    # LOGIC
    # --------------------------

    def select_analysis(self, analysis_type):

        self.state["analysis_type"] = analysis_type

        self.pages.setCurrentIndex(1)

    def run_analysis(self):

        print(self.state)

        # You have full access to napari here:
        print(self.viewer.layers)


viewer = napari.Viewer()

tool = MyTool(viewer)

viewer.window.add_dock_widget(
    tool,
    name="FluoroFate",
    area="right",
)

napari.run()